<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1 prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

In [1]:
# # This cell is for papermill parameters. DO NOT CHANGE THE VARIABLE NAMES.
# # Default values for manual execution (papermill will override these)
# input_geojson_path = None
# output_csv_path = None

## Step 1.1: Configure the environment

In [2]:
import os
from datetime import datetime
from shapely.geometry import Polygon
from shapely import box
from pyproj import CRS 
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
import rioxarray
from ipyleaflet import basemaps
from numpy.lib.stride_tricks import sliding_window_view
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import planetary_computer
# from odc.stac import load
from pystac.client import Client
from skimage.feature import graycomatrix, graycoprops
from utils import load_data, load_s1_dem, scale, calculate_band_indices, apply_mask, mask_water, all_masks, do_prediction

In [3]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

In [4]:
# Predefined variable for title and version

# Enter your initials
initials = "mw"

# Enter your site name
site = "samoa"

# Date
date = datetime.now()

# Make a clean version string
version = f"{initials}-{site}-{date.strftime('%d%m%Y')}_postcard_4"
print(version)

mw-samoa-27022026_postcard_4


In [5]:
gdfs = []
postcards_path = "training-data/"
file_extension: str = ".geojson"

for filename in os.listdir(postcards_path):
    file_path = os.path.join(postcards_path, filename)
    if os.path.isfile(file_path) and filename.endswith(file_extension):
    # try:
        gdf = gpd.read_file(file_path)
        gdfs.append(gdf)

In [6]:
for filename in os.listdir(postcards_path):
    file_path = os.path.join(postcards_path, filename)
    if os.path.isfile(file_path) and filename.endswith(file_extension):
        print(filename) # This line will print the name of each GeoJSON file
        # The rest of your code to read the file and append to gdfs
        # gdf = gpd.read_file(file_path)
        # gdfs.append(gdf)

print("\nFinished listing GeoJSON files.")

mw-samoa-27022026_postcard_4.geojson
mw-samoa-27022026.geojson

Finished listing GeoJSON files.


## Step 1.2: Configure STAC access and search parameters

In [7]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

In [8]:
training = gpd.read_file(f"training-data/{version}.geojson")
training = training.to_crs("EPSG:4326")
min_lon, min_lat, max_lon, max_lat = training.total_bounds

bbox = [min_lon, min_lat, max_lon, max_lat]

In [9]:
training

,lulc_class,value_id,lulc_code,geometry
0,Settlements,5,5,POINT (-171.80608 -13.82372)
1,Settlements,5,5,POINT (-171.8146 -13.82825)
2,Settlements,5,5,POINT (-171.81379 -13.82703)
3,Settlements,5,5,POINT (-171.8214 -13.82675)
4,Settlements,5,5,POINT (-171.80683 -13.8275)
...,...,...,...,...
393,Grassland,3,3,POINT (-172.10402 -13.84666)
394,Grassland,3,3,POINT (-172.10385 -13.84695)
395,Grassland,3,3,POINT (-172.10818 -13.84091)
396,Grassland,3,3,POINT (-172.10788 -13.8411)


In [10]:
print(training['lulc_class'].value_counts())
print('total gps points',(len(training)))

lulc_class
Settlements    92
Land           79
Forest         72
Grassland      55
Cropland       50
Wetlands       50
Name: count, dtype: int64
total gps points 398


In [11]:
datetime = "2023"

items = client.search(
    collections=["dep_s2_geomad"],
    datetime=datetime,
    bbox=bbox
).item_collection()

print(f"Found {len(items)} items in for {datetime}")

Found 4 items in for 2023


In [12]:
measurements = ["nir", "red", "blue", "green", "emad", "smad", "bcmad", "green", "nir08", "nir09", "swir16", "swir22", "coastal", "rededge1", "rededge2", "rededge3"]
data = load_data(
    items,
    measurements,
    bbox,
)
    
# Now you can use the 'data' variable
print(data)

<xarray.Dataset> Size: 4GB
Dimensions:      (y: 6745, x: 14955, time: 1)
Coordinates:
  * y            (y) float64 54kB -1.501e+06 -1.501e+06 ... -1.569e+06
  * x            (x) float64 120kB 4.142e+06 4.142e+06 ... 4.292e+06 4.292e+06
    spatial_ref  int32 4B 3832
  * time         (time) datetime64[ns] 8B 2023-01-01
Data variables: (12/15)
    nir          (time, y, x) uint16 202MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) uint16 202MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    blue         (time, y, x) uint16 202MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) uint16 202MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    emad         (time, y, x) float32 403MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    smad         (time, y, x) float32 403MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    ...           ...
    swir16       (time, y, x) uint16 202MB d

In [13]:
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit='16GB')
configure_s3_access(cloud_defaults=True, requester_pays=True)

In [14]:
scaled = scale(data)
scaled = scaled.compute().squeeze()

2026-02-27 04:58:22,188 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 10.54 GiB -- Worker memory limit: 14.90 GiB


In [15]:
# Explore the site we are working on
# scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

In [16]:
scaled = calculate_band_indices(scaled)
scaled

<xarray.Dataset> Size: 25GB
Dimensions:        (y: 6745, x: 14955)
Coordinates:
  * y              (y) float64 54kB -1.501e+06 -1.501e+06 ... -1.569e+06
  * x              (x) float64 120kB 4.142e+06 4.142e+06 ... 4.292e+06 4.292e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2023-01-01
Data variables: (12/32)
    nir            (y, x) float64 807MB 0.019 0.018 0.0185 ... 0.4303 0.4275
    red            (y, x) float64 807MB 0.0204 0.0202 0.0213 ... 0.024 0.0254
    blue           (y, x) float64 807MB 0.0476 0.0459 0.0456 ... 0.024 0.0257
    green          (y, x) float64 807MB 0.0271 0.0263 0.0272 ... 0.0479 0.0494
    emad           (y, x) float32 403MB 0.04724 0.04681 ... 0.04814 0.04799
    smad           (y, x) float32 403MB 2.861e-06 3.287e-06 ... 4.072e-08
    ...             ...
    ndci           (y, x) float64 807MB 0.02158 0.03118 0.0116 ... 0.5572 0.5436
    nbi            (y, x) float64 807MB -0.0411 -0.01124 ... -0.4341 -0.431
    ndmi           (y, x) float64 807MB 0.0411 0.01124 0.01648 ... 0.4341 0.431
    bsi            (y, x) float64 807MB -0.2919 -0.2742 ... -0.6523 -0.6444
    awei           (y, x) float64 807MB 0.06463 0.0623 ... -0.7385 -0.7289
    tc_wetness     (y, x) float64 807MB 0.01812 0.0177 ... -0.09289 -0.09119

In [17]:
scaled, mask = all_masks(scaled, return_mask = True)
# mask.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

In [18]:
scaled

<xarray.Dataset> Size: 25GB
Dimensions:        (y: 6745, x: 14955)
Coordinates:
  * y              (y) float64 54kB -1.501e+06 -1.501e+06 ... -1.569e+06
  * x              (x) float64 120kB 4.142e+06 4.142e+06 ... 4.292e+06 4.292e+06
    spatial_ref    int32 4B 3832
    time           datetime64[ns] 8B 2023-01-01
Data variables: (12/32)
    nir            (y, x) float64 807MB nan nan nan nan ... 0.4236 0.4303 0.4275
    red            (y, x) float64 807MB nan nan nan nan ... 0.0263 0.024 0.0254
    blue           (y, x) float64 807MB nan nan nan nan ... 0.0271 0.024 0.0257
    green          (y, x) float64 807MB nan nan nan nan ... 0.0489 0.0479 0.0494
    emad           (y, x) float32 403MB nan nan nan ... 0.04754 0.04814 0.04799
    smad           (y, x) float32 403MB nan nan nan ... 3.502e-08 4.072e-08
    ...             ...
    ndci           (y, x) float64 807MB nan nan nan nan ... 0.5283 0.5572 0.5436
    nbi            (y, x) float64 807MB nan nan nan ... -0.4301 -0.4341 -0.431
    ndmi           (y, x) float64 807MB nan nan nan nan ... 0.4301 0.4341 0.431
    bsi            (y, x) float64 807MB nan nan nan ... -0.6419 -0.6523 -0.6444
    awei           (y, x) float64 807MB nan nan nan ... -0.7213 -0.7385 -0.7289
    tc_wetness     (y, x) float64 807MB nan nan nan ... -0.09289 -0.09119

In [ ]:
# scaled.odc.explore(vmin=0, vmax=0.3, bands=["red", "green", "blue"], crs="EPSG:3832", name=site)

### Postcard csv

The objective of this notebook was to train the machine learning model that will allow us to classify an area with land cover classes defined through the training data.

Step 1.2. Input the training data to sample geomad data from the postcard

In [19]:
# Reproject training data to the GeoMAD CRS and convert to xarray
training_reprojected = training.to_crs(scaled.odc.crs)
training_da = training_reprojected.assign(
    x=training_reprojected.geometry.x, y=training_reprojected.geometry.y
).to_xarray()

# Extract training values from the masked dataset
training_values = (
    scaled.sel(training_da[["x", "y"]], method="nearest")
    .squeeze()
    .compute()
    .to_pandas()
)
training_values

,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,swir16,...,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x,spatial_ref,time
index,,,,,,,,,,,,,,,,,,,,,
0,0.5859,0.5971,0.5806,0.6032,0.087675,1.593828e-08,0.000002,0.6082,0.5093,0.6194,...,0.029500,0.027794,-0.027794,-0.001288,0.422250,0.174120,-1543795.0,4251725.0,3832,2023-01-01
1,0.4178,0.4419,0.4672,0.4557,0.225469,1.460910e-07,0.000008,0.4066,0.3904,0.4664,...,-0.112258,0.054965,-0.054965,0.012938,0.396725,0.036713,-1544315.0,4250775.0,3832,2023-01-01
2,0.3451,0.1273,0.1354,0.1021,0.068352,1.778841e-07,0.000003,0.3477,0.2732,0.1682,...,0.430807,-0.344633,0.344633,0.103377,-0.263300,-0.090507,-1544175.0,4250865.0,3832,2023-01-01
3,0.2732,0.1185,0.1725,0.1348,0.082369,1.745343e-07,0.000005,0.2939,0.2795,0.2119,...,0.141615,-0.126366,0.126366,-0.104585,-0.157450,-0.064524,-1544145.0,4250025.0,3832,2023-01-01
4,0.3521,0.3850,0.4085,0.4062,0.350881,9.259283e-08,0.000011,0.4286,0.3897,0.6096,...,0.118993,0.267755,-0.267755,0.174024,0.155475,-0.097274,-1544225.0,4251645.0,3832,2023-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393,0.3545,0.1075,0.0808,0.1235,0.119589,4.241317e-07,0.000006,0.3688,0.4119,0.2795,...,0.272173,-0.118297,0.118297,-0.187236,-0.513825,-0.077104,-1546415.0,4218565.0,3832,2023-01-01
394,0.3705,0.0813,0.0629,0.1068,0.120873,4.034221e-07,0.000006,0.3992,0.2651,0.2625,...,0.384557,-0.170616,0.170616,-0.283958,-0.579500,-0.072142,-1546445.0,4218585.0,3832,2023-01-01
395,0.4024,0.0946,0.0801,0.1271,0.120185,3.391653e-07,0.000006,0.4485,0.2922,0.2733,...,0.282246,-0.191061,0.191061,-0.315431,-0.576575,-0.072995,-1545755.0,4218095.0,3832,2023-01-01


In [20]:
training_values.columns

Index(['nir', 'red', 'blue', 'green', 'emad', 'smad', 'bcmad', 'nir08',
       'nir09', 'swir16', 'swir22', 'coastal', 'rededge1', 'rededge2',
       'rededge3', 'mndwi', 'ndti', 'cai', 'ndvi', 'evi', 'savi', 'ndwi',
       'b_g', 'b_r', 'swir22_swir16', 'mci', 'ndci', 'nbi', 'ndmi', 'bsi',
       'awei', 'tc_wetness', 'y', 'x', 'spatial_ref', 'time'],
      dtype='object')

In [21]:
# Join the training data with the extracted values and remove unnecessary columns
training_array = pd.concat([training["lulc_code"], training_values], axis=1)

# Drop rows where there was no data available
training_array = training_array.dropna()

# Preview our resulting training array
training_array.head()

,lulc_code,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,...,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x,spatial_ref,time
0,5,0.5859,0.5971,0.5806,0.6032,0.087675,1.593828e-08,0.000002,0.6082,0.5093,...,0.029500,0.027794,-0.027794,-0.001288,0.422250,0.174120,-1543795.0,4251725.0,3832,2023-01-01
1,5,0.4178,0.4419,0.4672,0.4557,0.225469,1.460910e-07,0.000008,0.4066,0.3904,...,-0.112258,0.054965,-0.054965,0.012938,0.396725,0.036713,-1544315.0,4250775.0,3832,2023-01-01
2,5,0.3451,0.1273,0.1354,0.1021,0.068352,1.778841e-07,0.000003,0.3477,0.2732,...,0.430807,-0.344633,0.344633,0.103377,-0.263300,-0.090507,-1544175.0,4250865.0,3832,2023-01-01
3,5,0.2732,0.1185,0.1725,0.1348,0.082369,1.745343e-07,0.000005,0.2939,0.2795,...,0.141615,-0.126366,0.126366,-0.104585,-0.157450,-0.064524,-1544145.0,4250025.0,3832,2023-01-01
4,5,0.3521,0.3850,0.4085,0.4062,0.350881,9.259283e-08,0.000011,0.4286,0.3897,...,0.118993,0.267755,-0.267755,0.174024,0.155475,-0.097274,-1544225.0,4251645.0,3832,2023-01-01


In [22]:
print(training_array.shape[1], 'total columns')
print('columns included', training_array.columns)

37 total columns
columns included Index(['lulc_code', 'nir', 'red', 'blue', 'green', 'emad', 'smad', 'bcmad',
       'nir08', 'nir09', 'swir16', 'swir22', 'coastal', 'rededge1', 'rededge2',
       'rededge3', 'mndwi', 'ndti', 'cai', 'ndvi', 'evi', 'savi', 'ndwi',
       'b_g', 'b_r', 'swir22_swir16', 'mci', 'ndci', 'nbi', 'ndmi', 'bsi',
       'awei', 'tc_wetness', 'y', 'x', 'spatial_ref', 'time'],
      dtype='object')


In [23]:
print(training_array['lulc_code'].value_counts())
print('total gps points',(len(training_array)))

lulc_code
5    92
1    72
6    64
3    54
2    50
4    46
Name: count, dtype: int64
total gps points 378


In [24]:
training_array=training_array.drop(columns=["spatial_ref", "time"])

In [25]:
training_array

,lulc_code,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,...,swir22_swir16,mci,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x
0,5,0.5859,0.5971,0.5806,0.6032,0.087675,1.593828e-08,0.000002,0.6082,0.5093,...,0.914433,0.925008,0.029500,0.027794,-0.027794,-0.001288,0.422250,0.174120,-1543795.0,4251725.0
1,5,0.4178,0.4419,0.4672,0.4557,0.225469,1.460910e-07,0.000008,0.4066,0.3904,...,0.999786,1.184576,-0.112258,0.054965,-0.054965,0.012938,0.396725,0.036713,-1544315.0,4250775.0
2,5,0.3451,0.1273,0.1354,0.1021,0.068352,1.778841e-07,0.000003,0.3477,0.2732,...,2.758621,1.078437,0.430807,-0.344633,0.344633,0.103377,-0.263300,-0.090507,-1544175.0,4250865.0
3,5,0.2732,0.1185,0.1725,0.1348,0.082369,1.745343e-07,0.000005,0.2939,0.2795,...,1.145824,1.733503,0.141615,-0.126366,0.126366,-0.104585,-0.157450,-0.064524,-1544145.0,4250025.0
4,5,0.3521,0.3850,0.4085,0.4062,0.350881,9.259283e-08,0.000011,0.4286,0.3897,...,1.141896,0.720041,0.118993,0.267755,-0.267755,0.174024,0.155475,-0.097274,-1544225.0,4251645.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393,3,0.3545,0.1075,0.0808,0.1235,0.119589,4.241317e-07,0.000006,0.3688,0.4119,...,0.681574,1.886642,0.272173,-0.118297,0.118297,-0.187236,-0.513825,-0.077104,-1546415.0,4218565.0
394,3,0.3705,0.0813,0.0629,0.1068,0.120873,4.034221e-07,0.000006,0.3992,0.2651,...,0.611048,2.025697,0.384557,-0.170616,0.170616,-0.283958,-0.579500,-0.072142,-1546445.0,4218585.0
395,3,0.4024,0.0946,0.0801,0.1271,0.120185,3.391653e-07,0.000006,0.4485,0.2922,...,0.572631,2.381065,0.282246,-0.191061,0.191061,-0.315431,-0.576575,-0.072995,-1545755.0,4218095.0
396,3,0.3983,0.0920,0.0768,0.1261,0.178438,4.424453e-07,0.000008,0.4442,0.3829,...,0.670225,1.786099,0.415873,-0.083073,0.083073,-0.198083,-0.654700,-0.107656,-1545775.0,4218135.0


In [26]:
# Write the training data to a CSV file
training_array.to_csv(f"training-data/{version}-training.csv", index=False)

In [27]:
training_array["lulc_code"].dtype

dtype('int32')